<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ crime clustering ★ </h3>
  <p> This notebook uses various clustering methods for nested clustering of crime. It produces a cluster csv and a folium map displaying the clusters. </p>
   <p> Do not modify the first cell. Different clustering methods can be specified in the cell after that. </p>
</div>

In [4]:
TIME = "_all"

In [ ]:
# imports

import math
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import Birch, OPTICS, KMeans
from sklearn.mixture import GaussianMixture
import folium
import matplotlib.pyplot as plt
import colorsys
from dataclasses import dataclass
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from shapely import wkt
import numpy as np
import re
import folium
import ast
import branca
from shapely import LineString, MultiLineString
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import os
import pathlib
from pathlib import Path
import math
from sklearn.neighbors import KNeighborsRegressor
import json

# clustering adapters for diff methods

@dataclass
class ClustererFactory:
    """Factory descriptor with an indicator for whether it supports k-based search."""
    make: callable         # make(k) -> model with .fit_predict(X)
    supports_k: bool       # True if k-based (Birch/GMM/KMeans), False for k-less (OPTICS)

def birch_factory(threshold=0.5, branching_factor=50) -> ClustererFactory:
    def _make(k: int):
        return Birch(n_clusters=k, threshold=threshold, branching_factor=branching_factor)
    return ClustererFactory(make=_make, supports_k=True)

def gmm_factory(random_state=0, reg_covar=1e-6) -> ClustererFactory:
    class GMMAdapter:
        def __init__(self, k):
            self.gmm = GaussianMixture(n_components=k, random_state=random_state, reg_covar=reg_covar)
        def fit_predict(self, X):
            self.gmm.fit(X)
            return self.gmm.predict(X)
    def _make(k: int):
        return GMMAdapter(k)
    return ClustererFactory(make=_make, supports_k=True)

def kmeans_factory(random_state=0, n_init=10, max_iter=300) -> ClustererFactory:
    def _make(k: int):
        return KMeans(n_clusters=k, random_state=random_state, n_init=n_init, max_iter=max_iter)
    return ClustererFactory(make=_make, supports_k=True)

# min samples needed to form a cluster, xi to control sensitivity/reliability of cluster split, min cluster size to filter clusters for visualization
def optics_factory(min_samples=10, xi=0.02, min_cluster_size=0.01) -> ClustererFactory:
    """Non-k-based example: runs once; silhouette k-search is skipped with a note."""
    class OPTICSAdapter:
        def __init__(self, _k_ignored):
            self.model = OPTICS(min_samples=min_samples, xi=xi, min_cluster_size=min_cluster_size)
        def fit_predict(self, X):
            return self.model.fit_predict(X)
    def _make(_k: int):
        return OPTICSAdapter(_k)
    return ClustererFactory(make=_make, supports_k=False)

# helpers
def to_datetime_safe(s):
    return pd.to_datetime(s, errors='coerce')

# calculate recency in score
def recency_weight(days, decay_lambda):
    days = np.asarray(days)
    days = np.clip(days, a_min=0, a_max=None)
    return np.exp(-decay_lambda * days)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = phi2 - phi1
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

# get cluster area
def bbox_area_km2(latitudes, longitudes):
    """Bounding-box area (km²) as a simple hotspot size proxy."""
    if len(latitudes) == 0:
        return 0.0
    lat_min, lat_max = float(np.min(latitudes)), float(np.max(latitudes))
    lon_min, lon_max = float(np.min(longitudes)), float(np.max(longitudes))
    height_km = haversine_km(lat_min, lon_min, lat_max, lon_min)
    width_km  = haversine_km(lat_min, lon_min, lat_min, lon_max)
    return max(height_km * width_km, 0.0)

# silhouette to find optimal k
def silhouette_k_search(X_scaled, k_min, k_max, factory: ClustererFactory, print_prefix="k"):
    """
    Try k in [k_min, k_max], compute silhouette, return (best_score, best_k, best_labels).
    If factory.supports_k is False, run once (k is ignored) and return (None, 1, labels).
    """
    n = len(X_scaled)
    if not factory.supports_k:
        print(f"{print_prefix}: non-k method; skipping silhouette k-search")
        labels = factory.make(1).fit_predict(X_scaled)
        return (None, 1, labels)

    if n < 3:
        labels = factory.make(1).fit_predict(X_scaled)
        return (None, 1, labels)

    best = None  # (score, k, labels)
    any_valid = False
    for k in range(max(2, k_min), min(k_max, n) + 1):
        model = factory.make(k)
        labels = model.fit_predict(X_scaled)
        if len(np.unique(labels)) < 2:
            continue
        try:
            s = silhouette_score(X_scaled, labels)
            any_valid = True
            print(f"{print_prefix}={k:2d}  silhouette={s:.3f}")
            if (best is None) or (s > best[0]):
                best = (s, k, labels)
        except Exception:
            continue

    if any_valid:
        return best
    else:
        labels = factory.make(1).fit_predict(X_scaled)
        return (None, 1, labels)

# color helpers
def hex_to_rgb01(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2],16)/255 for i in (0,2,4))

def rgb01_to_hex(rgb):
    r,g,b = (int(round(255*x)) for x in rgb)
    return f'#{r:02x}{g:02x}{b:02x}'

def shift_lightness(hex_color, lightness):
    r,g,b = hex_to_rgb01(hex_color)
    h,l,s = colorsys.rgb_to_hls(r,g,b)
    r2,g2,b2 = colorsys.hls_to_rgb(h, lightness, s)
    return rgb01_to_hex((r2,g2,b2))

# main function for clustering
def get_method_factories(method: str):
    """
    Returns (outer_factory, inner_factory, method_label_for_UI).
    method: 'birch' | 'gmm' | 'kmeans' | 'optics'
    """
    m = method.strip().lower()
    if m == "birch":
        return birch_factory(), birch_factory(), "Birch"
    if m == "gmm":
        return gmm_factory(), gmm_factory(), "GMM"
    if m == "kmeans":
        return kmeans_factory(), kmeans_factory(), "K-Means"
    if m == "optics":
        return optics_factory(), optics_factory(), "OPTICS"
    raise ValueError(f"Unknown method '{method}'. Use 'birch', 'gmm', 'kmeans', or 'optics'.")

def run_pipeline(
    method: str = "birch",
    night_path: str = "data/crime_points/crime_nighttime.csv",
    day_path: str   = "data/crime_points/crime_daytime.csv",
    csv_out_path: str | None = None,
    map_out_path: str | None = None,
    lon_col: str = "Longitude",
    lat_col: str = "Latitude",
    time_col: str = "Offense DateTime",
    severity_col: str = "Crime Score",
    max_score: float = 10.0,
    decay_lambda: float = 0.01,
    k_min_outer: int = 3,
    k_max_outer: int = 15,
    k_inner_min: int = 1,
    k_inner_max: int = 15,
    area_eps_km2: float = 1e-5,
    preserve_legacy_birch_col: bool = True,
):
    """
    Full workflow: loads data, chooses k (if applicable), clusters outer+inner, scores clusters,
    draws folium map, writes CSV & HTML.

    - method: 'birch' (default), 'gmm', 'kmeans', or 'optics'
    - preserve_legacy_birch_col: if True and method='birch', also writes df['birch_cluster'].
    """

    # outputs default to method-specific filenames (keeps old names when method='birch')
    if csv_out_path is None:
        csv_out_path = (
            f"data/clusters/nested_birch_cluster_scores{TIME}.csv"
            if method.lower() == "birch" else
            f"data/clusters/nested_{method.lower()}_cluster_scores{TIME}.csv"
        )
    if map_out_path is None:
        map_out_path = (
            f"data/clusters/nested_birch_shaded_map{TIME}.html"
            if method.lower() == "birch" else
            f"data/clusters/nested_{method.lower()}_shaded_map{TIME}.html"
        )

    OUTER, INNER, METHOD_LABEL = get_method_factories(method)

    if TIME == "_day":
        df = pd.read_csv(day_path)
    elif TIME == "_night":
        df = pd.read_csv(night_path) 
    else:
        night_df = pd.read_csv(night_path)
        day_df = pd.read_csv(day_path)
        df = pd.concat([night_df, day_df], ignore_index=True)

    # validate
    missing = [c for c in [lon_col, lat_col, time_col, severity_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # feature + outer clustering
    X = df[[lon_col, lat_col]].values
    X_scaled = StandardScaler().fit_transform(X)

    best_outer = silhouette_k_search(
        X_scaled,
        k_min=k_min_outer, k_max=k_max_outer,
        factory=OUTER,
        print_prefix="k"
    )
    _, best_k_outer, outer_labels = best_outer
    df['outer_cluster'] = outer_labels
    if preserve_legacy_birch_col and method.lower() == "birch":
        df['birch_cluster'] = outer_labels

    # check size
    _ = df.groupby('outer_cluster').size()

    # calculate recency and severity of crime
    now = pd.Timestamp.now()
    df[time_col] = to_datetime_safe(df[time_col])
    df['delta_days']   = (now - df[time_col]).dt.days
    df['recency']      = recency_weight(df['delta_days'].fillna(0), decay_lambda)
    df['severity_raw'] = pd.to_numeric(df[severity_col], errors='coerce').fillna(0.0)
    df['severity_norm'] = (df['severity_raw'] / float(max_score)).clip(0, 1)

    # perform nested clustering for each outer cluster
    df['nested_cluster'] = -1
    next_nested_id = 0
    for outer_label in np.unique(df['outer_cluster']):
        sub = df[df['outer_cluster'] == outer_label]
        if len(sub) == 0:
            continue

        X_inner = sub[[lon_col, lat_col]].to_numpy()
        X_inner_scaled = StandardScaler().fit_transform(X_inner)

        best_inner = silhouette_k_search(
            X_inner_scaled,
            k_min=k_inner_min, k_max=k_inner_max,
            factory=INNER,
            print_prefix=f"inner (outer={outer_label}) k"
        )
        _, k_chosen, inner_labels = best_inner

        # map local ids to global nested ids
        mapping = {loc_id: (next_nested_id + i) for i, loc_id in enumerate(np.unique(inner_labels))}
        df.loc[sub.index, 'nested_cluster'] = [mapping[i] for i in inner_labels]
        next_nested_id += len(mapping)

    # get score for each cluster
    records = []
    for nid, g in df.groupby('nested_cluster'):
        if nid == -1 or len(g) == 0:
            continue
        N = int(len(g))
        avg_score = g['severity_raw'].mean()  # 1..10
        S = float((g['recency'] * g['severity_norm']).sum())

        area_km2 = bbox_area_km2(g[lat_col].values, g[lon_col].values)
        divisor = max(area_km2, area_eps_km2)
        crime_weight = S * (N / divisor)

        # scale from 1-10 to 0.1-1
        avg_score01 = (avg_score - 1.0) / 9.0
        avg_score01 = float(np.clip(avg_score01, 0.0, 1.0))
        avg_score01_scaled = float(np.clip(avg_score01 * 0.9 + 0.1, 0.1, 1.0))

        records.append({
            'nested_cluster': nid,
            'outer_cluster': int(g['outer_cluster'].iloc[0]),
            'N': N,
            'avg_score': avg_score,
            'avg_score01_scaled': avg_score01_scaled,
            'crime_weight': crime_weight,
            'bbox_area_km2': area_km2,
            'centroid_lat': float(g[lat_col].mean()),
            'centroid_lon': float(g[lon_col].mean()),
        })

    # write data to cluster csv
    cluster_scores = pd.DataFrame.from_records(records).sort_values('avg_score', ascending=False)
    print(cluster_scores.head())
    cluster_scores.to_csv(csv_out_path, index=False)

    # prepare for map
    outer_labels = np.sort(df['outer_cluster'].unique())
    base_palette = plt.cm.tab10(np.linspace(0, 1, max(10, len(outer_labels))))
    base_hex = [plt.matplotlib.colors.rgb2hex(c) for c in base_palette]
    outer_hue = {lab: base_hex[i % len(base_hex)] for i, lab in enumerate(outer_labels)}

    # normalize avg_score within each outer cluster to 0-1
    cs = cluster_scores.copy()
    cs['avg_score'] = pd.to_numeric(cs['avg_score'], errors='coerce').fillna(0.0)
    norm_rows = []
    for outer in outer_labels:
        sub = cs[cs['outer_cluster'] == outer]
        if len(sub) == 0:
            continue
        w_min, w_max = sub['avg_score'].min(), sub['avg_score'].max()
        span = (w_max - w_min) if (w_max > w_min) else 1.0
        shade = (sub['avg_score'] - w_min) / span
        norm_rows.append(pd.DataFrame({
            'nested_cluster': sub['nested_cluster'].values,
            'outer_cluster': sub['outer_cluster'].values,
            'shade01': shade.values
        }))
    norm_df = pd.concat(norm_rows, ignore_index=True)
    shade_map = {row['nested_cluster']: (row['outer_cluster'], row['shade01'])
                 for _, row in norm_df.iterrows()}

    # color per nested cluster
    nested_color = {}
    for nid, (outer, shade) in shade_map.items():
        base = outer_hue[outer]
        L_safe, L_risky = 0.85, 0.35
        L = L_safe + (L_risky - L_safe) * (1.0 - float(shade))
        nested_color[nid] = shift_lightness(base, L)

    # folium map
    m = folium.Map(location=[df[lat_col].mean(), df[lon_col].mean()], zoom_start=13)

    for _, row in df.iterrows():
        nid = int(row['nested_cluster'])
        col = nested_color.get(nid, '#666666')
        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=3,
            color=col,
            fill=True,
            fill_opacity=0.9,
            weight=0
        ).add_to(m)

    for _, row in cluster_scores.iterrows():
        nid = int(row['nested_cluster'])
        outer = int(row['outer_cluster'])
        N = int(row['N'])
        lat, lon = float(row['centroid_lat']), float(row['centroid_lon'])
        col = nested_color.get(nid, '#444444')
        s01 = float(row['avg_score01_scaled'])

        folium.CircleMarker(
            location=[lat, lon],
            radius=6 + 0.2 * np.sqrt(N),
            color='black',
            weight=1,
            fill=True,
            fill_color=col,
            fill_opacity=0.95,
            popup=folium.Popup(
                html=(
                    f"<b>Nested:</b> {nid}<br>"
                    f"<b>{METHOD_LABEL} (outer):</b> {outer}<br>"
                    f"<b>N:</b> {N}<br>"
                    f"<b>Average Cluster Score:</b> {s01:,.2f}"
                ),
                max_width=280
            )
        ).add_to(m)

    # legend for clusters
    legend_html = """
    <div style="position: fixed; bottom: 12px; left: 12px; z-index: 9999; background: white; padding: 10px; border:1px solid #999; font-size:12px;">
    <b>Clusters</b><br>
    """ + "<br>".join(
        f'<span style="display:inline-block;width:12px;height:12px;background:{outer_hue[lab]};margin-right:6px;border:1px solid #555;"></span>'
        f'{METHOD_LABEL} {lab}'
        for lab in outer_labels
    ) + "</div>"
    m.get_root().html.add_child(folium.Element(legend_html))

    m.save(map_out_path)
    print(f"Saved map → {map_out_path}")
    print(f"Saved scores → {csv_out_path}")

    return {
        "df": df,
        "cluster_scores": cluster_scores,
        "map_path": map_out_path,
        "csv_path": csv_out_path,
        "method": METHOD_LABEL,
        "outer_k": best_k_outer
    }


In [6]:

# run diff clustering methods
# art_birch  = run_pipeline(method="birch") # saves .../nested_birch_shaded_map.html
# art_gmm    = run_pipeline(method="gmm") # saves .../nested_gmm_shaded_map.html
art_kmeans = run_pipeline(method="kmeans") # saves .../nested_kmeans_shaded_map.html
# art_optics = run_pipeline(method="optics") # saves .../nested_optics_shaded_map.html



k= 3  silhouette=0.459
k= 4  silhouette=0.452
k= 5  silhouette=0.434
k= 6  silhouette=0.442
k= 7  silhouette=0.429
k= 8  silhouette=0.415
k= 9  silhouette=0.426
k=10  silhouette=0.426
k=11  silhouette=0.424
k=12  silhouette=0.418
k=13  silhouette=0.426
k=14  silhouette=0.425
k=15  silhouette=0.421
inner (outer=0) k= 2  silhouette=0.386
inner (outer=0) k= 3  silhouette=0.449
inner (outer=0) k= 4  silhouette=0.454
inner (outer=0) k= 5  silhouette=0.444
inner (outer=0) k= 6  silhouette=0.446
inner (outer=0) k= 7  silhouette=0.433
inner (outer=0) k= 8  silhouette=0.424
inner (outer=0) k= 9  silhouette=0.424
inner (outer=0) k=10  silhouette=0.426
inner (outer=0) k=11  silhouette=0.400
inner (outer=0) k=12  silhouette=0.428
inner (outer=0) k=13  silhouette=0.427
inner (outer=0) k=14  silhouette=0.414
inner (outer=0) k=15  silhouette=0.404
inner (outer=1) k= 2  silhouette=0.489
inner (outer=1) k= 3  silhouette=0.457
inner (outer=1) k= 4  silhouette=0.465
inner (outer=1) k= 5  silhouette=0.461

In [13]:
def plot_score_map(edges, col):
    gdf = gpd.GeoDataFrame(
        edges,
        geometry=gpd.GeoSeries.from_wkt(edges["geometry"]),
        crs="EPSG:4326"
    )
    bounds = gdf.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2

    vmin = gdf[col].min()
    vmax = gdf[col].max()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="cartodbpositron")

    colormap = branca.colormap.LinearColormap(
        colors=["red", "orange", "yellow", "green"],
        vmin=vmin,
        vmax=vmax,
        caption="Crime (Density-based) Score"
    )
    colormap.add_to(m)

    # loop through each road segment
    for _, row in gdf.iterrows():
        geom = row.geometry

        # extract linestrings
        if isinstance(geom, LineString):
            lines = [geom]
        elif isinstance(geom, MultiLineString):
            lines = list(geom.geoms)
        else:
            continue

        # tooltip changed for crime
        # get the length
        length_val = row.get("length", None)
        if length_val is not None and not pd.isna(length_val):
            length_text = f"{length_val:.1f}"
        else:
            length_text = "N/A"

        tooltip_html = f"""
        <b>OSM ID:</b> {row.osmid}<br>
        <b>{col}</b> {row[col]:.3f}<br>
        <b>Length (m):</b> {length_text}<br>
        """
        # draw each linestring
        for line in lines:
            coords = [(pt[1], pt[0]) for pt in line.coords]
            folium.PolyLine(
                locations=coords,
                color=colormap(row[col]),
                weight=4,
                opacity=0.8,
                tooltip=folium.Tooltip(tooltip_html, sticky=True)
            ).add_to(m)

    # add the colored legend/bar
    colormap.add_to(m)

    m.save(f"map.html")

In [ ]:
def compute_edge_latlon(df):
    """
    Return df with guaranteed columns 'lat' and 'lon' (edge representative point).
    Tries several schemas; falls back to parsing 'geometry' if present.
    """
    df = df.copy()

    # 1) Already present
    if {'lat','lon'}.issubset(df.columns):
        return df

    # 2) Common alternatives
    colmap = {
        ('Latitude','Longitude'): ('lat','lon'),
        ('latitude','longitude'): ('lat','lon'),
        ('y','x'): ('lat','lon'),
        ('Y','X'): ('lat','lon')
    }
    for (src_lat, src_lon), (dst_lat, dst_lon) in colmap.items():
        if {src_lat, src_lon}.issubset(df.columns):
            df[dst_lat] = pd.to_numeric(df[src_lat], errors='coerce')
            df[dst_lon] = pd.to_numeric(df[src_lon], errors='coerce')
            return df

    # 3) Endpoints → midpoint
    endpoint_sets = [
        ('y1','x1','y2','x2'),
        ('lat_u','lon_u','lat_v','lon_v'),
        ('u_lat','u_lon','v_lat','v_lon'),
        ('lat1','lon1','lat2','lon2'),
        ('y_u','x_u','y_v','x_v')
    ]
    for a,b,c,d in endpoint_sets:
        if {a,b,c,d}.issubset(df.columns):
            df['lat'] = (pd.to_numeric(df[a], errors='coerce') + pd.to_numeric(df[c], errors='coerce'))/2
            df['lon'] = (pd.to_numeric(df[b], errors='coerce') + pd.to_numeric(df[d], errors='coerce'))/2
            return df

    # 4) Geometry column (WKT or shapely)
    if 'geometry' in df.columns:
        try:
            from shapely import wkt
            from shapely.geometry import LineString, MultiLineString, Point
            def geom_to_latlon(g):
                # parse WKT strings
                if isinstance(g, str):
                    try:
                        g = wkt.loads(g)
                    except Exception:
                        return np.nan, np.nan
                if isinstance(g, Point):
                    return g.y, g.x
                if isinstance(g, LineString):
                    mid = g.interpolate(0.5, normalized=True)
                    return mid.y, mid.x
                if isinstance(g, MultiLineString):
                    # take the longest line, then midpoint
                    line = max(g.geoms, key=lambda L: L.length)
                    mid = line.interpolate(0.5, normalized=True)
                    return mid.y, mid.x
                return np.nan, np.nan

            lats, lons = zip(*df['geometry'].map(geom_to_latlon))
            df['lat'] = lats
            df['lon'] = lons
            return df
        except Exception:
            pass  # fall through

    # 5) Still nothing — help the user see what's available
    raise KeyError(
        "Could not infer lat/lon. Available columns: "
        f"{list(df.columns)}. Provide lat/lon, endpoints, or a parsable 'geometry' column."
    )



In [ ]:
# --- 1) Load cluster centroids + avg_score (crime-only) ---
# This CSV must have: centroid_lat, centroid_lon, avg_score (crime mean per nested cluster)
clusters = pd.read_csv("data/clusters/nested_kmeans_cluster_scores_all_test.csv")  # or your chosen method/file

# Sanity: keep only needed cols and drop na
clusters = clusters[["centroid_lat", "centroid_lon", "avg_score"]].dropna()

# --- 2) Normalize avg_score to [0,1] ---
# If your crime scores are on 1..15, set these; otherwise we can fall back to min/max from the data.
FIXED_MIN, FIXED_MAX = 1.0, 15.0   # adjust if you intentionally used a different scale
if clusters["avg_score"].between(FIXED_MIN, FIXED_MAX).all():
    denom = max(FIXED_MAX - FIXED_MIN, 1e-9)
    clusters["avg01"] = np.clip((clusters["avg_score"] - FIXED_MIN) / denom, 0, 1)
else:
    # data-driven fallback
    a, b = clusters["avg_score"].min(), clusters["avg_score"].max()
    denom = max(b - a, 1e-9)
    clusters["avg01"] = np.clip((clusters["avg_score"] - a) / denom, 0, 1)

# --- 3) Train KNN on cluster centroids to predict edge risk ---
X = clusters[["centroid_lat", "centroid_lon"]].to_numpy()
y = clusters["avg01"].to_numpy()

knn = KNeighborsRegressor(n_neighbors=7, weights="distance")  # keep your 7-NN default
knn.fit(X, y)

# --- 4) Ensure edges have representative lat/lon and predict ---
edges = pd.read_csv("data/edges/edges_seattle_filtered.csv")

def compute_edge_latlon(df):
    df = df.copy()
    # fast paths
    if {'lat','lon'}.issubset(df.columns): return df
    for a,b in [('Latitude','Longitude'), ('latitude','longitude'), ('y','x'), ('Y','X')]:
        if {a,b}.issubset(df.columns):
            df['lat'] = pd.to_numeric(df[a], errors='coerce')
            df['lon'] = pd.to_numeric(df[b], errors='coerce')
            return df
    for a,b,c,d in [('y1','x1','y2','x2'), ('lat_u','lon_u','lat_v','lon_v'),
                    ('u_lat','u_lon','v_lat','v_lon'), ('lat1','lon1','lat2','lon2'),
                    ('y_u','x_u','y_v','x_v')]:
        if {a,b,c,d}.issubset(df.columns):
            df['lat'] = (pd.to_numeric(df[a], errors='coerce') + pd.to_numeric(df[c], errors='coerce'))/2
            df['lon'] = (pd.to_numeric(df[b], errors='coerce') + pd.to_numeric(df[d], errors='coerce'))/2
            return df
    if 'geometry' in df.columns:
        from shapely import wkt
        from shapely.geometry import LineString, MultiLineString, Point
        def geom_to_latlon(g):
            if isinstance(g, str):
                try: g = wkt.loads(g)
                except Exception: return np.nan, np.nan
            if isinstance(g, Point): return g.y, g.x
            if isinstance(g, LineString):
                mid = g.interpolate(0.5, normalized=True); return mid.y, mid.x
            if isinstance(g, MultiLineString):
                L = max(g.geoms, key=lambda L: L.length)
                mid = L.interpolate(0.5, normalized=True); return mid.y, mid.x
            return np.nan, np.nan
        lats, lons = zip(*df['geometry'].map(geom_to_latlon))
        df['lat'], df['lon'] = lats, lons
        return df
    raise KeyError("Need lat/lon, endpoints, or a parsable 'geometry' column to locate edges.")

edges = compute_edge_latlon(edges)
coords = edges[["lat","lon"]].to_numpy()
edges["risk_score"] = np.clip(knn.predict(coords), 0, 1)

edges.to_csv("edges_with_risk.csv", index=False)
print("Wrote edges_with_risk.csv with risk_score ∈ [0,1]")

# --- 5) Plot with your fixed 0–1 colormap ---
plot_score_map(edges, "risk_score")



Wrote edges_with_risk.csv with risk_score ∈ [0,1]


In [ ]:
plot_score_map(edges, "risk_score")

In [15]:
# cluster_scores from your pipeline
cluster_scores = pd.read_csv("data/clusters/nested_kmeans_cluster_scores_all_test.csv")
lowest = cluster_scores.loc[cluster_scores['avg_score'].idxmin()]

print("Lowest risk cluster:")
print(lowest)


Lowest risk cluster:
nested_cluster     11.000000
outer_cluster       2.000000
N                 281.000000
avg_score           4.320285
avg_score01         0.237163
centroid_lat       47.605475
centroid_lon     -122.338260
Name: 20, dtype: float64
